# Generalização (item 2.2) — versão Python

Versão Python de [`analysis/02-2_generalizacao.qmd`](../analysis/02-2_generalizacao.qmd). No notebook de modelagem (`01_modelagem.ipynb`) já usei train/test split estratificado e 5-fold CV pra escolher `max_features`. Aqui vou além, respondendo três perguntas que a validação básica não responde:

1. **O RMSE de teste é um número exato ou tem incerteza?** → bootstrap.
2. **O split aleatório está sendo generoso demais?** → split temporal.
3. **O modelo generaliza para um CEP que nunca viu?** → validação cruzada agrupada por `zipcode`.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA = "../data"
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Reconstruindo o dataset e o split (mesma preparação de `01_modelagem.ipynb`)

Repito a preparação de dados e uso o mesmo `random_state=42` e a mesma estratificação — então `X_train`/`X_test` aqui são exatamente o mesmo split usado para treinar e avaliar o modelo salvo, permitindo reaproveitá-lo sem retreinar.

In [2]:
casas = pd.read_csv(f"{DATA}/kc_house_data.csv")
casas["date"] = pd.to_datetime(casas["date"], format="%Y%m%dT%H%M%S")
casas = casas[casas["bedrooms"] < 30].copy()

demograf = pd.read_csv(f"{DATA}/zipcode_demographics.csv")

casas["log_price"] = np.log(casas["price"])
casas["tem_porao"] = (casas["sqft_basement"] > 0).astype(int)
casas["idade_casa"] = casas["date"].dt.year - casas["yr_built"]
casas["reformado"] = (casas["yr_renovated"] > 0).astype(int)
casas = casas.merge(demograf, on="zipcode", how="left")

vars_modelo = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "grade", "tem_porao", "idade_casa", "reformado",
    "lat", "long", "sqft_living15", "sqft_lot15",
    "medn_hshld_incm_amt", "medn_incm_per_prsn_amt", "hous_val_amt", "per_bchlr", "per_prfsnl"]

X = casas[vars_modelo]
y = casas["log_price"]
bins = pd.qcut(y, q=10, labels=False, duplicates="drop")

idx_treino, idx_teste = train_test_split(casas.index, test_size=0.2, random_state=42, stratify=bins)
casas_treino = casas.loc[idx_treino]
casas_teste = casas.loc[idx_teste]

def metricas(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return pd.Series({"rmse": rmse, "mae": mae, "r2": r2})

modelo_rf = joblib.load("models/modelo_rf.joblib")
pred_teste = modelo_rf.predict(casas_teste[vars_modelo])
real_teste = casas_teste["log_price"].values
metricas(real_teste, pred_teste)

rmse   0.1692
mae    0.1200
r2     0.8978
dtype: float64

## 2. Quanta incerteza tem o RMSE de teste?

*Bootstrap*: reamostro as mesmas previsões de teste, com reposição, 2.000 vezes, recalculando a métrica a cada vez — isso não usa dado novo, usa a variação natural de "que teste eu teria tido se o sorteio tivesse saído diferente" para construir um intervalo em vez de um único número.

In [3]:
rng = np.random.default_rng(123)
n_teste = len(real_teste)
boot_rmse = np.empty(2000)
boot_mae = np.empty(2000)

for b in range(2000):
    idx_boot = rng.integers(0, n_teste, n_teste)
    boot_rmse[b] = np.sqrt(mean_squared_error(real_teste[idx_boot], pred_teste[idx_boot]))
    boot_mae[b] = mean_absolute_error(real_teste[idx_boot], pred_teste[idx_boot])

pd.DataFrame({
    "metrica": ["rmse", "mae"],
    "estimado_pontual": [np.sqrt(mean_squared_error(real_teste, pred_teste)), mean_absolute_error(real_teste, pred_teste)],
    "ic_95_inferior": [np.percentile(boot_rmse, 2.5), np.percentile(boot_mae, 2.5)],
    "ic_95_superior": [np.percentile(boot_rmse, 97.5), np.percentile(boot_mae, 97.5)],
})

,metrica,estimado_pontual,ic_95_inferior,ic_95_superior
0,rmse,0.1692,0.1629,0.1759
1,mae,0.1200,0.1166,0.1236


Intervalo estreito — o RMSE de teste é estável, não é sorte de um sorteio específico de teste.

## 3. Split aleatório vs. split temporal

Um split aleatório mistura vendas "do futuro" e "do passado" entre treino e teste — algo que nunca vai acontecer numa produção real, onde sempre se prevê o futuro a partir do passado. Refaço o split ordenando por data e usando as primeiras 80% das vendas como treino.

Uso `max_features` e `min_samples_leaf` iguais ao modelo final, mas `n_estimators=200` (mais rápido) nos dois lados desta comparação, pra que a única diferença entre as duas linhas seja a forma do split, não o hiperparâmetro.

In [4]:
casas_ordenado = casas.sort_values("date")
corte = int(0.8 * len(casas_ordenado))
treino_temporal = casas_ordenado.iloc[:corte]
teste_temporal = casas_ordenado.iloc[corte:]

print("treino:", treino_temporal["date"].min().date(), "a", treino_temporal["date"].max().date())
print("teste: ", teste_temporal["date"].min().date(), "a", teste_temporal["date"].max().date())

max_features_final = modelo_rf.max_features

rf_temporal = RandomForestRegressor(
    n_estimators=200, max_features=max_features_final, min_samples_leaf=5, random_state=42, n_jobs=-1
).fit(treino_temporal[vars_modelo], treino_temporal["log_price"])
pred_temporal = rf_temporal.predict(teste_temporal[vars_modelo])

rf_aleatorio_ref = RandomForestRegressor(
    n_estimators=200, max_features=max_features_final, min_samples_leaf=5, random_state=42, n_jobs=-1
).fit(casas_treino[vars_modelo], casas_treino["log_price"])
pred_aleatorio_ref = rf_aleatorio_ref.predict(casas_teste[vars_modelo])

pd.DataFrame({
    "split temporal": metricas(teste_temporal["log_price"], pred_temporal),
    "split aleatório (ref, n_estimators=200)": metricas(casas_teste["log_price"], pred_aleatorio_ref),
}).T

treino: 2014-05-02 a 2015-03-10
teste:  2015-03-10 a 2015-05-27


,rmse,mae,r2
split temporal,0.1894,0.1391,0.8668
"split aleatório (ref, n_estimators=200)",0.1691,0.1200,0.8978


O split temporal piora o RMSE em relação ao aleatório — confirma com número real a preocupação de que parte do desempenho reportado vem de o teste ter imóveis "intercalados" no tempo com o treino, algo que não se repete em produção.

## 4. E se aparecer um CEP que o modelo nunca viu?

Diferente de uma CV comum (que sorteia *linhas*, deixando o mesmo CEP presente nos dois lados), sorteio *zipcodes inteiros* em 5 grupos com `GroupKFold` — cada fold testa em CEPs 100% ausentes do treino daquele fold. Isso simula "apareceu uma casa num bairro que o modelo nunca viu".

In [5]:
gkf = GroupKFold(n_splits=5)
resultados_grupo = []

for fold, (idx_tr, idx_te) in enumerate(gkf.split(X, y, groups=casas["zipcode"])):
    rf_g = RandomForestRegressor(
        n_estimators=200, max_features=max_features_final, min_samples_leaf=5, random_state=42, n_jobs=-1
    ).fit(X.iloc[idx_tr], y.iloc[idx_tr])
    pred_g = rf_g.predict(X.iloc[idx_te])
    m = metricas(y.iloc[idx_te], pred_g)
    resultados_grupo.append({
        "fold": fold, "rmse": m["rmse"], "mae": m["mae"], "r2": m["r2"],
        "n_zipcodes_teste": casas["zipcode"].iloc[idx_te].nunique(), "n_obs_teste": len(idx_te),
    })

resultados_grupo = pd.DataFrame(resultados_grupo)
resultados_grupo

,fold,rmse,mae,r2,n_zipcodes_teste,n_obs_teste
0,0,0.2421,0.1800,0.8015,14,4339
1,1,0.2204,0.1634,0.8619,14,4325
2,2,0.2074,0.1523,0.8049,14,4306
3,3,0.1865,0.1360,0.8372,14,4343
4,4,0.1823,0.1343,0.8801,14,4299


In [6]:
resultados_grupo[["rmse", "mae", "r2"]].agg(["mean", "std"])

,rmse,mae,r2
mean,0.2077,0.1532,0.8371
std,0.0247,0.0192,0.0345


## 5. Síntese

In [7]:
sintese = pd.DataFrame({
    "split aleatório (modelo final)": metricas(real_teste, pred_teste),
    "split aleatório (n_estimators=200, referência)": metricas(casas_teste["log_price"], pred_aleatorio_ref),
    "split temporal": metricas(teste_temporal["log_price"], pred_temporal),
    "CV agrupada por CEP (média dos 5 folds)": resultados_grupo[["rmse", "mae", "r2"]].mean(),
}).T
sintese["variacao_rmse_vs_aleatorio"] = sintese["rmse"] / sintese.loc["split aleatório (modelo final)", "rmse"] - 1
sintese

,rmse,mae,r2,variacao_rmse_vs_aleatorio
split aleatório (modelo final),0.1692,0.1200,0.8978,0.0000
"split aleatório (n_estimators=200, referência)",0.1691,0.1200,0.8978,-0.0000
split temporal,0.1894,0.1391,0.8668,0.1197
CV agrupada por CEP (média dos 5 folds),0.2077,0.1532,0.8371,0.2282


O padrão se repete de forma consistente com a versão R: **a maior ameaça à generalização não é o tempo, é o bairro novo** — o RMSE da CV agrupada por CEP piora bem mais que o do split temporal, e essa mesma ordem (CEP novo > tempo) apareceu de forma independente nas duas linguagens, com implementações e bibliotecas diferentes. Isso reforça que não é um artefato de uma ferramenta específica — é uma característica real dos dados e das features escolhidas (não usar `zipcode` como categoria, só `lat`/`long` + demografia).